# LLM Evaluation and Fine-tuning

**MIDAS AI in Research Handbook — Chapter 26**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/lora_finetuning_demo.ipynb)

---

This notebook walks through a complete evaluation and fine-tuning workflow using arXiv abstract classification as a running example. The task is to assign each abstract to one of several computer science and mathematics subject areas based only on its text.

Working through the four steps in order mirrors the decision logic in the chapter: evaluate the out-of-the-box model first, understand the cost of full fine-tuning, apply LoRA as a practical middle ground, and then compare the results.

**What you need:** A Google account to run in Colab. Enable the GPU by going to Runtime → Change runtime type → T4 GPU. Training takes about eight minutes on a T4.

## Step 1 — Install Libraries

This takes about two to three minutes on a fresh Colab session. Run it once and then proceed.

In [ ]:
!pip install -q transformers datasets peft accelerate evaluate scikit-learn
print("Installation complete.")

## Step 2 — Check for GPU

The cell below detects whether a GPU is available and sets a `device` variable that subsequent cells use. If no GPU is found, a warning is printed and the notebook falls back to CPU, but training will take significantly longer.

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    device_name = torch.cuda.get_device_name(0)
    print(f"GPU available: {device_name}")
else:
    device = torch.device("cpu")
    print("No GPU found. Training on CPU will be slow. Consider enabling a GPU runtime.")

print(f"Using device: {device}")

---

## Part 1 — Load the Dataset and Establish a Baseline

### Load the arXiv Classification Dataset

The dataset contains arXiv research abstracts labeled with subject-area categories. We work with a five-category subset covering Artificial Intelligence, Computation and Language, Computer Vision, Mathematical Optimization, and Quantitative Biology. This subset is large enough to train a meaningful model but small enough to finish quickly on a free GPU.

In [ ]:
from datasets import load_dataset
import pandas as pd

# Load the dataset from the Hugging Face Hub.
# ccdv/arxiv-classification contains abstracts for 11 arXiv subject categories.
# We use a 5-category subset for clarity.
raw = load_dataset("ccdv/arxiv-classification", trust_remote_code=True)

# The dataset uses integer labels 0-10. We select five categories.
# Original label mapping (from the dataset card):
# 0: cs.AI  1: cs.CL  2: cs.CV  3: cs.DS  4: cs.NE
# 5: cs.PL  6: cs.SE  7: math-ph  8: math.CO  9: math.IT  10: math.NT
# We keep: 0 (cs.AI), 1 (cs.CL), 2 (cs.CV), 9 (math.IT), 4 (cs.NE)
# and remap to 0-4 for the classification head.

KEEP_LABELS = {0: 0, 1: 1, 2: 2, 4: 3, 9: 4}
LABEL_NAMES = {
    0: "Artificial Intelligence",
    1: "Computation and Language",
    2: "Computer Vision",
    3: "Neural and Evolutionary Computing",
    4: "Information Theory",
}
NUM_LABELS = len(LABEL_NAMES)

def filter_and_remap(example):
    return example["label"] in KEEP_LABELS

def remap_label(example):
    example["label"] = KEEP_LABELS[example["label"]]
    return example

train_raw = raw["train"].filter(filter_and_remap).map(remap_label)
test_raw  = raw["test"].filter(filter_and_remap).map(remap_label)

# Cap training set at 5000 examples total and test at 1000 for speed.
# Shuffle with a fixed seed so results are reproducible.
train_data = train_raw.shuffle(seed=42).select(range(min(5000, len(train_raw))))
test_data  = test_raw.shuffle(seed=42).select(range(min(1000, len(test_raw))))

print(f"Training examples : {len(train_data)}")
print(f"Test examples     : {len(test_data)}")
print(f"Number of classes : {NUM_LABELS}")
print()
print("Class distribution in training set:")
from collections import Counter
counts = Counter(train_data["label"])
for k, v in sorted(counts.items()):
    print(f"  {LABEL_NAMES[k]:<35} {v:>5}")

### Explore a Few Examples

Read a couple of abstracts before doing anything computational. Getting familiar with the text is always the right first step.

In [ ]:
for i in [0, 1, 2]:
    label = train_data[i]["label"]
    text  = train_data[i]["text"]
    print(f"--- Example {i+1} | Label: {LABEL_NAMES[label]} ---")
    print(text[:400])
    print()

---

### Out-of-the-Box Baseline: Zero-Shot Classification

Before touching any training code, we measure what a general-purpose model can do without any task-specific training. We use a BART-based zero-shot classifier and supply the five category names as candidate labels. This model has never seen these specific abstracts and has no knowledge of the label definitions beyond their plain-English names.

Zero-shot classification with `facebook/bart-large-mnli` works by framing each classification as a natural-language inference problem: it asks whether the abstract text is consistent with each candidate label description, and the label with the highest entailment score wins.

This will take about three to four minutes to run on the full test set. We evaluate on the first 200 test examples to keep inference time manageable.

In [ ]:
from transformers import pipeline
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# Load zero-shot classifier. Move to GPU if available.
zs_device = 0 if torch.cuda.is_available() else -1
zero_shot = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=zs_device,
)

candidate_labels = list(LABEL_NAMES.values())

# Evaluate on a 200-example slice so this finishes quickly.
eval_slice = test_data.select(range(200))

print("Running zero-shot inference on 200 test examples...")
zs_preds = []
for example in eval_slice:
    result = zero_shot(example["text"][:512], candidate_labels)
    pred_label_name = result["labels"][0]
    pred_int = candidate_labels.index(pred_label_name)
    zs_preds.append(pred_int)

true_labels = [ex["label"] for ex in eval_slice]

zs_accuracy = accuracy_score(true_labels, zs_preds)
print(f"
Zero-shot accuracy on 200 test examples: {zs_accuracy:.3f}")
print()
print(classification_report(
    true_labels, zs_preds,
    target_names=candidate_labels,
    zero_division=0,
))

The accuracy you see here is the starting point. Notice which categories the zero-shot model struggles with. Categories that share vocabulary with other fields tend to be confused more often, while categories with distinctive terminology are easier. This pattern is diagnostic: it shows where the model's general-purpose knowledge runs out.

A five-class random baseline would achieve 20% accuracy. Zero-shot performance meaningfully above that shows the model is learning something from the label names, but there is usually a substantial gap between zero-shot performance and what a task-specific model can achieve.

---

## Part 2 — The Cost of Full Fine-tuning

Before jumping to fine-tuning, it is worth making the cost concrete. The cell below loads the base model we will use later (DistilBERT) and counts its parameters. Then it estimates what full fine-tuning would require in terms of memory and time.

In [ ]:
from transformers import AutoModelForSequenceClassification
import math

model_name = "distilbert-base-uncased"
base_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=NUM_LABELS,
    ignore_mismatched_sizes=True,
)

total_params = sum(p.numel() for p in base_model.parameters())
trainable_params = sum(p.numel() for p in base_model.parameters() if p.requires_grad)

print(f"Model: {model_name}")
print(f"Total parameters       : {total_params:>12,}")
print(f"Trainable parameters   : {trainable_params:>12,}")
print()

# Memory estimate for full fine-tuning:
# Each parameter stored as float32 needs 4 bytes.
# During training, you also need gradients (4 bytes each) and
# Adam optimizer states (8 bytes each, two momentum terms).
# Rough estimate: ~16 bytes per parameter total.
bytes_per_param = 16
total_bytes = total_params * bytes_per_param
total_gb = total_bytes / (1024 ** 3)

print(f"Estimated GPU memory for full fine-tuning:")
print(f"  ~{total_gb:.1f} GB for model + gradients + optimizer states")
print()

# For a 7B parameter model (e.g., Llama-2-7B):
large_model_params = 7_000_000_000
large_model_gb = (large_model_params * bytes_per_param) / (1024 ** 3)
print(f"For comparison, a 7-billion-parameter model:")
print(f"  ~{large_model_gb:.0f} GB — requires multiple high-memory GPUs")
print()
print("DistilBERT is small enough that full fine-tuning is feasible.")
print("With models 10x or 100x larger, this calculation is why most")
print("researchers cannot afford full fine-tuning without institutional HPC access.")

DistilBERT is on the small end of modern language models. It was designed to be lightweight, and full fine-tuning is feasible here. But the numbers scale quickly. For a model with several billion parameters, the memory required for full fine-tuning can easily exceed what a single A100 GPU (80 GB) can hold, requiring multi-GPU setups, gradient checkpointing, and mixed-precision training, all of which add engineering complexity.

There is also the question of data. Full fine-tuning on small datasets often leads to overfitting, because all the model's weights are being adjusted on too few examples. LoRA sidesteps part of this problem by drastically reducing the number of parameters that can overfit.

---

## Part 3 — LoRA Fine-tuning with PEFT

### Configure the LoRA Adapter

We define a `LoraConfig` and wrap the base model. The key parameters are `r` (the rank of the low-rank matrices, controlling adapter capacity) and `target_modules` (which layers get adapters). For DistilBERT, the query and value projection layers in the self-attention blocks are the standard targets.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,                          # rank of the low-rank matrices
    lora_alpha=32,                 # scaling factor (often set to 2*r)
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],  # DistilBERT attention layers
    bias="none",
)

# Re-initialize the base model (in case the cell above modified it)
base_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=NUM_LABELS,
    ignore_mismatched_sizes=True,
)

peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

The output of `print_trainable_parameters()` shows what LoRA actually does: instead of training all 66 million parameters in DistilBERT, we train roughly 370,000, which is less than one percent of the total. The frozen parameters still participate in the forward pass. They contribute their pretrained knowledge. The adapter matrices on top are what learn the task-specific adjustment.

### Tokenize the Data

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=256,
    )

# Tokenize and format for PyTorch
train_tok = train_data.map(tokenize, batched=True)
test_tok  = test_data.map(tokenize, batched=True)

train_tok = train_tok.remove_columns(
    [c for c in train_tok.column_names if c not in ["input_ids", "attention_mask", "label"]]
)
test_tok = test_tok.remove_columns(
    [c for c in test_tok.column_names if c not in ["input_ids", "attention_mask", "label"]]
)

train_tok = train_tok.rename_column("label", "labels")
test_tok  = test_tok.rename_column("label", "labels")
train_tok.set_format("torch")
test_tok.set_format("torch")

print(f"Train tokenized: {len(train_tok)} examples")
print(f"Test tokenized : {len(test_tok)} examples")

### Train the LoRA Adapter

We use the Hugging Face `Trainer` API, which handles the training loop, logging, and evaluation automatically. The only thing different from a standard fine-tuning setup is that `peft_model` has far fewer trainable parameters.

In [ ]:
from transformers import TrainingArguments, Trainer
import numpy as np
import evaluate as hf_evaluate

accuracy_metric = hf_evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=preds, references=labels)

training_args = TrainingArguments(
    output_dir="./lora-arxiv-output",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    evaluation_strategy="epoch",
    save_strategy="no",
    learning_rate=3e-4,           # LoRA adapters train well at higher LRs than full FT
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,
    fp16=torch.cuda.is_available(),  # mixed precision speeds up training on GPU
    report_to="none",
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    compute_metrics=compute_metrics,
)

print("Starting LoRA fine-tuning...")
print(f"Training on {len(train_tok)} examples for {training_args.num_train_epochs} epochs.")
print()
trainer.train()

---

## Part 4 — Compare Results

### Evaluate the Fine-tuned Model

We run the fine-tuned model on the same 200-example test slice used for the zero-shot baseline, so the comparison is on equal footing.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

peft_model.eval()
peft_model.to(device)

# Evaluate on the same 200-example slice as the zero-shot baseline
eval_slice_tok = test_tok.select(range(200))

import torch
from torch.utils.data import DataLoader

loader = DataLoader(eval_slice_tok, batch_size=64)

ft_preds = []
with torch.no_grad():
    for batch in loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        outputs = peft_model(input_ids=input_ids, attention_mask=attention_mask)
        preds = outputs.logits.argmax(dim=-1).cpu().numpy()
        ft_preds.extend(preds.tolist())

# true_labels was defined in Part 1 from the same 200-example slice
ft_accuracy = accuracy_score(true_labels, ft_preds)
print(f"Fine-tuned model accuracy on 200 test examples: {ft_accuracy:.3f}")
print()
print(classification_report(
    true_labels, ft_preds,
    target_names=candidate_labels,
    zero_division=0,
))

### Side-by-Side Summary

In [ ]:
print("=" * 55)
print(f"{'Method':<35}  {'Accuracy':>8}")
print("=" * 55)
print(f"{'Zero-shot (bart-large-mnli)':<35}  {zs_accuracy:>8.3f}")
print(f"{'LoRA fine-tuned (distilbert)':<35}  {ft_accuracy:>8.3f}")
print("=" * 55)
print()
print("Per-class breakdown — F1 scores:")
print(f"{'Category':<35}  {'Zero-shot':>9}  {'Fine-tuned':>10}")
print("-" * 58)

from sklearn.metrics import f1_score
for i, name in LABEL_NAMES.items():
    zs_f1 = f1_score(true_labels, zs_preds, labels=[i], average="macro", zero_division=0)
    ft_f1 = f1_score(true_labels, ft_preds, labels=[i], average="macro", zero_division=0)
    print(f"{name:<35}  {zs_f1:>9.3f}  {ft_f1:>10.3f}")

Look at which categories improved most and which least. Categories where the fine-tuned model still struggles are worth examining manually. Are the misclassified examples genuinely ambiguous, even to a domain expert? Or is the error systematic in a way that more labeled data might fix? That distinction matters when deciding whether to invest more in data collection.

### Save the LoRA Adapter

Unlike a full fine-tuned model, a LoRA adapter saves only the trainable matrices. The file is small enough to store alongside your code in a repository.

In [ ]:
import os

adapter_dir = "./lora-arxiv-adapter"
peft_model.save_pretrained(adapter_dir)

# Show what was saved and total size
total_size = 0
for fname in os.listdir(adapter_dir):
    fpath = os.path.join(adapter_dir, fname)
    size = os.path.getsize(fpath)
    total_size += size
    print(f"  {fname:<40} {size/1024:.1f} KB")

print(f"
Total adapter size: {total_size / (1024**2):.2f} MB")
print()
print("For comparison, the full DistilBERT model weights are about 260 MB.")
print("The adapter captures the task-specific adjustment in a fraction of that.")

---

## Exercise

Work through the following on your own.

1. Look at the per-class F1 table from Part 4. Pick the category with the largest gap between zero-shot and fine-tuned performance. Find five test examples that the zero-shot model got wrong but the fine-tuned model got right. What about those abstracts makes them difficult for a model with no task-specific training?

2. Try increasing `r` from 16 to 32 in the `LoraConfig` and retraining. Does accuracy improve? Look at `print_trainable_parameters()` to see how the parameter count changes. Is the trade-off worth it?

3. If you have a text classification problem from your own research, adapt the dataset loading and label definitions to your domain. The training code does not need to change. The more interesting work is building a hand-labeled reference set to evaluate against and deciding whether the performance you see is sufficient for your use case.

---

## Summary

| Step | What we did | Key finding |
|------|-------------|-------------|
| 1 — Baseline | Zero-shot classification with `bart-large-mnli` | General model struggles with domain-specific categories |
| 2 — Cost analysis | Counted parameters, estimated memory | Full fine-tuning of large models is expensive and often impractical |
| 3 — LoRA | Trained adapters (~0.5% of parameters) with PEFT | Training time under 10 minutes on T4 GPU |
| 4 — Comparison | Evaluated both models on same test slice | Substantial accuracy and F1 improvement with minimal compute |

**Next steps:** For larger experiments or sensitive data, move to Great Lakes. For more on model evaluation in research contexts, see Chapter 21 (Validation and Interpretation).